## 1. Introduction

Amazon hosts thousands of electronics products, each with different prices, discounts, ratings, and sales indicators. 
Understanding how these factors vary across products and categories can help optimize pricing, discount strategies, and product positioning.

In this notebook, I perform exploratory data analysis (EDA) on the 
**"Amazon Products Sales Dataset 42K+ Items – 2025"**, which contains over 42,000 
Amazon electronics products scraped from the marketplace, including features like product name, category, price, discounted price, discount percentage, ratings, review counts, and more.

**Objectives:**
- Understand the structure and quality of the raw dataset.
- Clean and preprocess the uncleaned CSV into an analysis‑ready dataset.
- Analyze pricing and discount distributions across products.
- Explore ratings and review patterns to understand product popularity.
- Identify top categories and sub‑categories by price, discount, and rating.


### 2. Setup & Data Loading

In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
df_raw= pd.read_csv('amazon_products_sales_data_uncleaned.csv')

df_raw.head()

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6 out of 5 stars,"35,882",10K+ bought in past month,NaN,basic variant price: $162.24,No Discount,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,/Apple-Cancellation-Transparency-Personalized-...,2025-08-21 11:14:29
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8 out of 5 stars,"28,988",10K+ bought in past month,NaN,basic variant price: $72.74,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...,2025-08-21 11:14:29


### 3. Data understanding

#### 3.1 Dataset Overview

- **Rows:** `df_raw.shape[0]` (number of products)
- **Columns:** `df_raw.shape[1]`
- All columns and data types are listed above.
- 
Here we look for object columns that should be numeric (prices, ratings, percentages) 
and any columns with many missing values.

In [2]:
df_raw.shape

(42675, 16)

#### Analysis:

We observe there are total of 4,265 rows and 16 columns present in `df_raw`

### 3.2 Sample rows and missing values

This sample shows how raw values are stored (e.g., currency symbols in prices, 
text in rating columns, long product descriptions). The missing‑value table 
helps decide which columns to clean, impute, or drop.

In [3]:
df_raw.sample(5, random_state=42)

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
12457,"Bose QuietComfort Earbuds II, Wireless, Blueto...",3.9 out of 5 stars,"15,351",100+ bought in past month,NaN,basic variant price: nan,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51RM+UZFwO...,/Bose-QuietComfort-Cancelling-Personalized-Can...,2025-08-24 22:11:19
354,Fujifilm QuickSnap Flash 400 One Time Use 35mm...,4.6 out of 5 stars,380,5K+ bought in past month,68.88,basic variant price: nan,$76.89,Amazon's,Organic,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/81kfPqxQL5...,/Fujifilm-QuickSnap-Flash-Disposable-Camera/dp...,2025-08-21 11:17:53
11680,Renata Watch Battery 321 0%Hg Mercury Free X 3,4.6 out of 5 stars,"1,533",600+ bought in past month,6.95,basic variant price: 3 Count (Pack of 1),No Discount,No Badge,Organic,No Coupon,Add to cart,"Delivery Tue, Sep 2",NaN,https://m.media-amazon.com/images/I/61uoafSNsv...,NaN,2025-08-24 22:05:10
36143,"Apple Mid 2015 iPad Mini 4 7.9-inch, Wi-Fi, 64...",3.9 out of 5 stars,483,200+ bought in past month,109.00,basic variant price: nan,No Discount,No Badge,Organic,No Coupon,NaN,"FREE delivery Thu, Sep 4Or fastest delivery To...",NaN,https://m.media-amazon.com/images/I/61MRDo+gSb...,/Apple-iPad-Wi-Fi-7-9-Inch-Tablet/dp/B01N8SQV1...,2025-08-30 00:17:55
34102,Spigen Slim Armor CS Designed for iPhone 13 Pr...,4.3 out of 5 stars,"2,685",400+ bought in past month,17.99,basic variant price: nan,$39.99,No Badge,Organic,No Coupon,Add to cart,"FREE delivery Wed, Sep 3 on $35 of items shipp...",NaN,https://m.media-amazon.com/images/I/51qjmN+SM7...,/Spigen-Slim-Armor-Designed-iPhone/dp/B096HBY1...,2025-08-29 11:37:22


In [4]:
df_raw.isna().sum().sort_values(ascending=False)

sustainability_badges       39267
buy_box_availability        14653
current/discounted_price    11749
delivery_details            11720
bought_in_last_month         3217
product_url                  2069
rating                       1024
number_of_reviews            1024
title                           0
price_on_variant                0
listed_price                    0
is_best_seller                  0
is_sponsored                    0
is_couponed                     0
image_url                       0
collected_at                    0
dtype: int64

### 4. Data cleaning & feature engineering

#### 4.1 Copy raw → working DataFrame

In [5]:
df= df_raw.copy()

#### 4.2 Formating Columns

In [6]:
# Reaming the columns for better usecase
rename_map = {
    "number_of_reviews": "num_reviews",
    "bought_in_last_month": "bought_last_month",
    "current/discounted_price": "current_discounted_price",
    "price_on_variant": "base_variant_price",
    "is_couponed": "has_coupon",
    "buy_box_availability": "buy_box_available",
    # others left unchanged
}

df = df_raw.rename(columns=rename_map)
df.columns


Index(['title', 'rating', 'num_reviews', 'bought_last_month',
       'current_discounted_price', 'base_variant_price', 'listed_price',
       'is_best_seller', 'is_sponsored', 'has_coupon', 'buy_box_available',
       'delivery_details', 'sustainability_badges', 'image_url', 'product_url',
       'collected_at'],
      dtype='str')

#### 4.3 Cleaning and transforming columns

In [7]:
## The 'Rating' column has texts which needs to be removed and the column type is converted to float
df['rating']=df['rating'].str.replace(' out of 5 stars','').astype(float)
df['rating']

0        4.6
1        4.3
2        4.6
3        4.6
4        4.8
        ... 
42670    5.0
42671    4.2
42672    4.3
42673    4.7
42674    4.4
Name: rating, Length: 42675, dtype: float64

In [8]:
# The "num_reviews" contained commas , which is to be removed and converted to Int
df["num_reviews"] = (
    df["num_reviews"]
    .str.replace(",", "", regex=False)
    .str.strip()
    .replace("", np.nan)
    .astype("Int64"))
df["num_reviews"]

0          375
1         2457
2         3044
3        35882
4        28988
         ...  
42670        1
42671       20
42672       57
42673     7102
42674       75
Name: num_reviews, Length: 42675, dtype: Int64

In [9]:

# This column represents the **current selling price** on Amazon, sometimes with commas and a dollar sign.  
# Stripped the `$` symbol and commas and convert it to `float`, which allows histogram and correlation analysis with other numeric features.
df["current_discounted_price"] = (
    df["current_discounted_price"]
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
    .replace("", np.nan)
    .astype(float))
df["current_discounted_price"]

0         89.68
1          9.99
2        314.00
3           NaN
4           NaN
          ...  
42670    195.99
42671     89.99
42672    899.99
42673     10.39
42674    419.99
Name: current_discounted_price, Length: 42675, dtype: float64

In [10]:
#The `base_variant_price` field includes helper text such as `"Basic variant price:"`, units like `"GHz"`, and a dollar sign.  
#Removing that text and units, then extract the first numeric pattern (e.g. from `"Basic variant price: $199.99"` to`199.99`), remove commas, and convert to `float`.  
# This gives a clean base price for the main product variant.

df["base_variant_price"] = (
        df["base_variant_price"]
        .str.replace("basic variant price:", "", case=False, regex=False)
        .str.replace("GHz", "", case=False, regex=False)
        .str.replace("$", "", regex=False)
        .str.extract(r"([\d.,]+)", expand=False)   # first numeric chunk
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace("", np.nan)
        .astype(float)   )
df["base_variant_price"]

0          2.40
1           NaN
2           NaN
3        162.24
4         72.74
          ...  
42670       NaN
42671     25.00
42672     30.00
42673       NaN
42674       NaN
Name: base_variant_price, Length: 42675, dtype: float64

In [12]:
df["listed_price"].unique()

<StringArray>
[    '$159.00',      '$15.99',     '$349.00', 'No Discount',      '$19.98',
      '$14.33',      '$17.99',      '$21.89',      '$20.00',      '$48.99',
 ...
   '$2,899.00',      '$16.97',   '$2,999.99',       '$9.10',   '$1,749.99',
      '$21.39',     '$599.95',     '$139.98',     '$149.98',     '$950.43']
Length: 911, dtype: str

In [13]:
# This is the list (original) price before any discount.  
# Removing `$` and commas and convert it to `float`,
# so I can compare it to `current_discounted_price` and compute discount amounts and percentages.
df["listed_price"] = (
        df["listed_price"]
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("No Discount","", regex=False)
        .str.strip()
        .replace("", np.nan)
        .astype(float))
df["listed_price"]

0         159.00
1          15.99
2         349.00
3            NaN
4            NaN
          ...   
42670        NaN
42671        NaN
42672    1099.99
42673      15.98
42674     499.99
Name: listed_price, Length: 42675, dtype: float64

In [14]:
 # The scraped values can be `"Best seller"`, `"No Badge"`, or similar.  
 # Normalized them into `"Yes"` or `"No"` via a small mapping function:  
 #  `"Best seller"` → `Yes`, `"No Badge"` or anything else → `No`.  
 #  This makes it easy to filter and compare Best Seller products vs others.
def map_best_seller(val):
        s = str(val).strip().lower()
        if s == "best seller":
            return "Yes"
        elif s == "no badge" or s == "nan":
            return "No"
        else:
            return "No"
df["is_best_seller"] = df["is_best_seller"].apply(map_best_seller)

s = df["has_coupon"].astype(str).str.strip()

# is_discounted column
df["is_discounted"] = np.where(
        s.str.contains("save", case=False) & s.str.contains("coupon", case=False),
        "Yes",
        "No")

# coupon_value column
# extract number before % and divide by 100
coupon_pct = (
        s.str.extract(r"(\d+)\s*%", expand=False)
         .astype(float)
    )
df["coupon_value"] = coupon_pct / 100.0
df.drop(columns=['has_coupon'], inplace=True)
df[[ "is_discounted", "coupon_value"]]

,is_discounted,coupon_value
0,Yes,0.15
1,No,NaN
2,No,NaN
3,No,NaN
4,No,NaN
...,...,...
42670,No,NaN
42671,Yes,NaN
42672,No,NaN
42673,No,NaN


In [15]:
# Using the `title` text,Classifying each product into one of three broad categories:  
#   - `"Laptop"` if the title mentions **laptop** terms such as `"laptop"`, `"macbook"`, `"notebook pc"`, or `"chromebook"`.  
#   - `"Phone"` if the title clearly refers to a **phone** such as `"iPhone"`, `"smartphone"`, `"mobile phone"`, `"Galaxy S"`, or `"Pixel phone"`.  
#   - `"Other Electronics"` for all remaining titles (mics, earbuds, cables, accessories, etc.).  
#   This simple grouping helps compare laptops vs phones vs other electronics segments in the dataset.

def simple_product_category(title: str) -> str:
    t = str(title).lower()

    # Laptop keywords
    if any(k in t for k in [
        "laptop",
        "macbook",
        "notebook pc",
        "chromebook"
    ]):
        return "Laptop"

    # Phone keywords
    if any(k in t for k in [
        "iphone",
        "android phone",
        "smartphone",
        "mobile phone",
        "cell phone",
        "galaxy s",
        "pixel phone"
    ]):
        return "Phone"

    # Fallback
    return "Other Electronics"
df["product_category"] = df["title"].apply(simple_product_category)
df["product_category"].value_counts()


product_category
Other Electronics    36240
Laptop                3930
Phone                 2505
Name: count, dtype: int64

In [16]:
# The original field contains strings like `"123+ bought in past month"` or `"6k bought in past month"`.  
#   Removing the trailing text `"+ bought in past month"`, then:  
#   - Extract the numeric part plus an optional `k` suffix.  
#   - Convert `"6k"` to `6000` (multiply by 1,000), and plain numbers like `"123"` to `123`.  
#   - Store the result as a nullable integer (`Int64`).  
#   This gives a clean numeric estimate of recent purchase volume for each product, useful as a demand signal.

df["bought_last_month"] = (
    df["bought_last_month"]
    .astype(str)
    .str.replace("+ bought in past month", "", regex=False)
    .str.strip()
    .pipe(lambda s: s.str.extract(r"(\d+\.?\d*)\s*([kK]?)", expand=True))
    .pipe(lambda df_extracted: (
        (df_extracted[0].replace("", pd.NA).astype(float) *
         df_extracted[1].str.lower().map({"k": 1000}).fillna(1))
        .round()
        .astype("Int64")
    ))
)
df["bought_last_month"]

0          300
1         6000
2         2000
3        10000
4        10000
         ...  
42670      100
42671      200
42672       50
42673      500
42674       50
Name: bought_last_month, Length: 42675, dtype: Int64

In [17]:
df.head()

,title,rating,num_reviews,bought_last_month,current_discounted_price,base_variant_price,listed_price,is_best_seller,is_sponsored,buy_box_available,delivery_details,sustainability_badges,image_url,product_url,collected_at,is_discounted,coupon_value,product_category
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375,300,89.68,2.40,159.00,No,Sponsored,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29,Yes,0.15,Phone
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457,6000,9.99,NaN,15.99,No,Sponsored,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29,No,NaN,Laptop
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044,2000,314.00,NaN,349.00,No,Sponsored,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29,No,NaN,Phone
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6,35882,10000,NaN,162.24,NaN,Yes,Organic,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,/Apple-Cancellation-Transparency-Personalized-...,2025-08-21 11:14:29,No,NaN,Other Electronics
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8,28988,10000,NaN,72.74,NaN,No,Organic,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...,2025-08-21 11:14:29,No,NaN,Phone


In [18]:
df.info()
df.describe(include="all")
df.isna().sum().sort_values(ascending=False)

<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   title                     42675 non-null  str    
 1   rating                    41651 non-null  float64
 2   num_reviews               41651 non-null  Int64  
 3   bought_last_month         32233 non-null  Int64  
 4   current_discounted_price  30926 non-null  float64
 5   base_variant_price        21298 non-null  float64
 6   listed_price              12311 non-null  float64
 7   is_best_seller            42675 non-null  str    
 8   is_sponsored              42675 non-null  str    
 9   buy_box_available         28022 non-null  str    
 10  delivery_details          30955 non-null  str    
 11  sustainability_badges     3408 non-null   str    
 12  image_url                 42675 non-null  str    
 13  product_url               40606 non-null  str    
 14  collected_at     

coupon_value                41614
sustainability_badges       39267
listed_price                30364
base_variant_price          21377
buy_box_available           14653
current_discounted_price    11749
delivery_details            11720
bought_last_month           10442
product_url                  2069
num_reviews                  1024
rating                       1024
is_discounted                   0
collected_at                    0
title                           0
image_url                       0
is_sponsored                    0
is_best_seller                  0
product_category                0
dtype: int64